# 10 — Model Evaluation


In [1]:
#Imports

from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from scipy import stats

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    cross_validate,
    learning_curve,
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

ROOT = Path.cwd().parent
FEATURE_DIR = ROOT / "artifacts" / "features"
MODEL_DIR = ROOT / "artifacts" / "models"
PREDICTION_DIR = ROOT / "artifacts" / "predictions"
CHART_DIR = ROOT / "artifacts" / "charts"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# Load Training ANd Test DAta

X_train = pd.read_parquet(
    FEATURE_DIR / "X_train_raw.parquet"
)

X_test = pd.read_parquet(
    FEATURE_DIR / "X_test_raw.parquet"
)

with open(
    FEATURE_DIR / "feature_metadata.json",
    "r",
    encoding="utf-8",
) as file:
    feature_metadata = json.load(file)

TARGET_COLUMN = feature_metadata["target_column"]
problem_type = feature_metadata["problem_type"]

y_train = pd.read_parquet(
    FEATURE_DIR / "y_train.parquet"
)[TARGET_COLUMN]

y_test = pd.read_parquet(
    FEATURE_DIR / "y_test.parquet"
)[TARGET_COLUMN]

saved_preprocessor = joblib.load(
    FEATURE_DIR / "preprocessor.joblib"
)

if problem_type != "classification":
    raise ValueError(
        "This notebook section is currently configured "
        "for the detected classification problem."
    )

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Problem type:", problem_type)



Training shape: (400, 5)
Test shape: (100, 5)
Problem type: classification


In [3]:
# CAndidate Models

candidate_models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
    ),
    "Histogram Gradient Boosting":
        HistGradientBoostingClassifier(
            max_iter=200,
            random_state=RANDOM_STATE,
        ),
    "ANN MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        early_stopping=True,
        validation_fraction=0.15,
        max_iter=500,
        random_state=RANDOM_STATE,
    ),
}

print("Robust-validation candidates:")

for model_name in candidate_models:
    print(model_name)

Robust-validation candidates:
Logistic Regression
Decision Tree
Random Forest
Gradient Boosting
Histogram Gradient Boosting
ANN MLP


In [4]:
#Repeated VAlidated Configuration

REPEATS = 5
FOLDS = 5

repeated_cv = RepeatedStratifiedKFold(
    n_splits=FOLDS,
    n_repeats=REPEATS,
    random_state=RANDOM_STATE,
)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
    "roc_auc": "roc_auc",
}

print("Folds:", FOLDS)
print("Repeats:", REPEATS)
print("Total validation evaluations:", FOLDS * REPEATS)

Folds: 5
Repeats: 5
Total validation evaluations: 25


In [5]:
#Run Repeaded Cross VAlidation

validation_rows = []
fold_scores = {}

for model_name, model in candidate_models.items():
    print(f"Evaluating: {model_name}")

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                clone(saved_preprocessor),
            ),
            (
                "model",
                clone(model),
            ),
        ]
    )

    scores = cross_validate(
        estimator=pipeline,
        X=X_train,
        y=y_train,
        cv=repeated_cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True,
        error_score="raise",
    )

    model_f1_scores = scores["test_f1_macro"]
    fold_scores[model_name] = model_f1_scores

    sample_count = len(model_f1_scores)
    standard_error = (
        model_f1_scores.std(ddof=1)
        / np.sqrt(sample_count)
    )

    confidence_interval = stats.t.interval(
        confidence=0.95,
        df=sample_count - 1,
        loc=model_f1_scores.mean(),
        scale=standard_error,
    )

    train_f1 = scores["train_f1_macro"].mean()
    validation_f1 = model_f1_scores.mean()

    validation_rows.append({
        "model": model_name,
        "evaluations": sample_count,
        "cv_accuracy_mean":
            scores["test_accuracy"].mean(),
        "cv_balanced_accuracy_mean":
            scores["test_balanced_accuracy"].mean(),
        "cv_f1_macro_mean": validation_f1,
        "cv_f1_macro_std":
            model_f1_scores.std(ddof=1),
        "cv_f1_macro_ci_lower":
            confidence_interval[0],
        "cv_f1_macro_ci_upper":
            confidence_interval[1],
        "cv_f1_weighted_mean":
            scores["test_f1_weighted"].mean(),
        "cv_roc_auc_mean":
            scores["test_roc_auc"].mean(),
        "train_f1_macro_mean": train_f1,
        "train_validation_gap":
            train_f1 - validation_f1,
    })

Evaluating: Logistic Regression
Evaluating: Decision Tree
Evaluating: Random Forest
Evaluating: Gradient Boosting
Evaluating: Histogram Gradient Boosting
Evaluating: ANN MLP


In [6]:
# Robust Leaderboard

robust_results_df = pd.DataFrame(
    validation_rows
)

robust_results_df = robust_results_df.sort_values(
    by=[
        "cv_f1_macro_ci_lower",
        "cv_balanced_accuracy_mean",
    ],
    ascending=False,
).reset_index(drop=True)

display(robust_results_df.round(4))

,model,evaluations,cv_accuracy_mean,cv_balanced_accuracy_mean,cv_f1_macro_mean,cv_f1_macro_std,cv_f1_macro_ci_lower,cv_f1_macro_ci_upper,cv_f1_weighted_mean,cv_roc_auc_mean,train_f1_macro_mean,train_validation_gap
0,Decision Tree,25,0.8380,0.8383,0.8325,0.0308,0.8198,0.8453,0.8389,0.8567,0.9530,0.1205
1,Random Forest,25,0.8345,0.8371,0.8294,0.0321,0.8161,0.8426,0.8355,0.9136,1.0000,0.1706
2,Gradient Boosting,25,0.8320,0.8304,0.8257,0.0296,0.8135,0.8379,0.8327,0.9097,1.0000,0.1743
3,Histogram Gradient Boosting,25,0.8090,0.8014,0.7999,0.0362,0.7850,0.8148,0.8089,0.9054,1.0000,0.2001
4,Logistic Regression,25,0.7760,0.7756,0.7689,0.0460,0.7499,0.7879,0.7773,0.8605,0.7821,0.0132
5,ANN MLP,25,0.7580,0.7277,0.7316,0.0566,0.7083,0.7550,0.7493,0.8549,0.7669,0.0352


In [7]:
# VAlidation Performance Visualization

figure = px.bar(
    robust_results_df,
    x="model",
    y="cv_f1_macro_mean",
    error_y="cv_f1_macro_std",
    color="cv_balanced_accuracy_mean",
    title="Repeated Cross-Validation Performance",
    labels={
        "cv_f1_macro_mean": "Mean macro F1",
        "model": "Model",
    },
)

figure.update_layout(
    xaxis_tickangle=-35
)

figure.show()

figure.write_html(
    CHART_DIR / "robust_model_validation.html"
)

In [8]:
# VAlidation Winner Selection

FINAL_MODEL_NAME = robust_results_df.iloc[0][
    "model"
]

print(
    "Validation-selected model:",
    FINAL_MODEL_NAME,
)

display(
    robust_results_df.iloc[[0]].round(4)
)

Validation-selected model: Decision Tree


,model,evaluations,cv_accuracy_mean,cv_balanced_accuracy_mean,cv_f1_macro_mean,cv_f1_macro_std,cv_f1_macro_ci_lower,cv_f1_macro_ci_upper,cv_f1_weighted_mean,cv_roc_auc_mean,train_f1_macro_mean,train_validation_gap
0,Decision Tree,25,0.838,0.8383,0.8325,0.0308,0.8198,0.8453,0.8389,0.8567,0.953,0.1205


In [9]:
# Paired STatistical Comparison

winner_scores = fold_scores[FINAL_MODEL_NAME]

statistical_comparisons = []

for model_name, model_scores in fold_scores.items():
    if model_name == FINAL_MODEL_NAME:
        continue

    score_differences = (
        winner_scores - model_scores
    )

    if np.allclose(score_differences, 0):
        statistic = 0.0
        p_value = 1.0
    else:
        statistic, p_value = stats.wilcoxon(
            winner_scores,
            model_scores,
            alternative="two-sided",
        )

    statistical_comparisons.append({
        "winner": FINAL_MODEL_NAME,
        "comparison_model": model_name,
        "winner_mean_f1":
            winner_scores.mean(),
        "comparison_mean_f1":
            model_scores.mean(),
        "mean_difference":
            score_differences.mean(),
        "wilcoxon_statistic": statistic,
        "raw_p_value": p_value,
    })

statistical_comparison_df = pd.DataFrame(
    statistical_comparisons
)

display(
    statistical_comparison_df.round(6)
)



,winner,comparison_model,winner_mean_f1,comparison_mean_f1,mean_difference,wilcoxon_statistic,raw_p_value
0,Decision Tree,Logistic Regression,0.83252,0.768918,0.063602,15.0,0.000008
1,Decision Tree,Random Forest,0.83252,0.829378,0.003142,117.0,0.757760
2,Decision Tree,Gradient Boosting,0.83252,0.825730,0.006790,144.0,0.618625
3,Decision Tree,Histogram Gradient Boosting,0.83252,0.799920,0.032600,31.0,0.000140
4,Decision Tree,ANN MLP,0.83252,0.731635,0.100885,3.0,0.000000


In [10]:
# Holm Multiple test Correction

comparison_count = len(
    statistical_comparison_df
)

sorted_indices = (
    statistical_comparison_df["raw_p_value"]
    .sort_values()
    .index
    .tolist()
)

previous_adjusted_value = 0.0
adjusted_values = {}

for rank, index in enumerate(
    sorted_indices,
    start=1,
):
    multiplier = comparison_count - rank + 1

    adjusted_value = min(
        statistical_comparison_df.loc[
            index,
            "raw_p_value",
        ] * multiplier,
        1.0,
    )

    adjusted_value = max(
        adjusted_value,
        previous_adjusted_value,
    )

    adjusted_values[index] = adjusted_value
    previous_adjusted_value = adjusted_value

statistical_comparison_df[
    "holm_adjusted_p_value"
] = pd.Series(adjusted_values)

statistical_comparison_df[
    "significant_at_0.05"
] = (
    statistical_comparison_df[
        "holm_adjusted_p_value"
    ] < 0.05
)

display(
    statistical_comparison_df.round(6)
)

,winner,comparison_model,winner_mean_f1,comparison_mean_f1,mean_difference,wilcoxon_statistic,raw_p_value,holm_adjusted_p_value,significant_at_0.05
0,Decision Tree,Logistic Regression,0.83252,0.768918,0.063602,15.0,0.000008,0.000033,True
1,Decision Tree,Random Forest,0.83252,0.829378,0.003142,117.0,0.757760,1.000000,False
2,Decision Tree,Gradient Boosting,0.83252,0.825730,0.006790,144.0,0.618625,1.000000,False
3,Decision Tree,Histogram Gradient Boosting,0.83252,0.799920,0.032600,31.0,0.000140,0.000420,True
4,Decision Tree,ANN MLP,0.83252,0.731635,0.100885,3.0,0.000000,0.000001,True


In [11]:
#Traing Final SElected Pipeline

final_model = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(saved_preprocessor),
        ),
        (
            "model",
            clone(
                candidate_models[FINAL_MODEL_NAME]
            ),
        ),
    ]
)

final_model.fit(
    X_train,
    y_train,
)

final_predictions = final_model.predict(
    X_test
)

final_probabilities = final_model.predict_proba(
    X_test
)[:, 1]

print("Final model fitted:", FINAL_MODEL_NAME)

Final model fitted: Decision Tree


In [12]:
# Holdout Evaluation

final_test_metrics = {
    "accuracy": accuracy_score(
        y_test,
        final_predictions,
    ),
    "balanced_accuracy":
        balanced_accuracy_score(
            y_test,
            final_predictions,
        ),
    "precision_macro":
        precision_score(
            y_test,
            final_predictions,
            average="macro",
            zero_division=0,
        ),
    "recall_macro":
        recall_score(
            y_test,
            final_predictions,
            average="macro",
            zero_division=0,
        ),
    "f1_macro":
        f1_score(
            y_test,
            final_predictions,
            average="macro",
            zero_division=0,
        ),
    "f1_weighted":
        f1_score(
            y_test,
            final_predictions,
            average="weighted",
            zero_division=0,
        ),
    "roc_auc": roc_auc_score(
        y_test,
        final_probabilities,
    ),
    "log_loss": log_loss(
        y_test,
        final_probabilities,
    ),
}

print("Final holdout metrics:")

for metric_name, value in final_test_metrics.items():
    print(f"{metric_name}: {value:.4f}")

Final holdout metrics:
accuracy: 0.8100
balanced_accuracy: 0.8119
precision_macro: 0.8011
recall_macro: 0.8119
f1_macro: 0.8043
f1_weighted: 0.8117
roc_auc: 0.8556
log_loss: 4.1084


In [13]:
#Classification Report

print(
    classification_report(
        y_test,
        final_predictions,
        zero_division=0,
    )
)

final_confusion_matrix = confusion_matrix(
    y_test,
    final_predictions,
)

final_confusion_df = pd.DataFrame(
    final_confusion_matrix,
    index=[
        f"Actual {label}"
        for label in sorted(y_test.unique())
    ],
    columns=[
        f"Predicted {label}"
        for label in sorted(y_test.unique())
    ],
)

display(final_confusion_df)

              precision    recall  f1-score   support

           0       0.73      0.82      0.77        39
           1       0.88      0.80      0.84        61

    accuracy                           0.81       100
   macro avg       0.80      0.81      0.80       100
weighted avg       0.82      0.81      0.81       100



,Predicted 0,Predicted 1
Actual 0,32,7
Actual 1,12,49


In [14]:
# Bootstrapped Confidence Intervals

BOOTSTRAP_ITERATIONS = 2000

random_generator = np.random.default_rng(
    RANDOM_STATE
)

bootstrap_f1_scores = []

y_test_array = y_test.to_numpy()
prediction_array = np.asarray(
    final_predictions
)

for _ in range(BOOTSTRAP_ITERATIONS):
    sample_indices = random_generator.integers(
        0,
        len(y_test_array),
        len(y_test_array),
    )

    sampled_actual = y_test_array[
        sample_indices
    ]

    sampled_predictions = prediction_array[
        sample_indices
    ]

    if len(np.unique(sampled_actual)) < 2:
        continue

    bootstrap_f1_scores.append(
        f1_score(
            sampled_actual,
            sampled_predictions,
            average="macro",
            zero_division=0,
        )
    )

bootstrap_lower = np.percentile(
    bootstrap_f1_scores,
    2.5,
)

bootstrap_upper = np.percentile(
    bootstrap_f1_scores,
    97.5,
)

print(
    "Holdout macro F1:",
    round(final_test_metrics["f1_macro"], 4),
)

print(
    "Bootstrap 95% interval:",
    (
        round(bootstrap_lower, 4),
        round(bootstrap_upper, 4),
    ),
)

Holdout macro F1: 0.8043
Bootstrap 95% interval: (np.float64(0.7182), np.float64(0.8796))


In [15]:
# Learning Curve

training_sizes = np.linspace(
    0.2,
    1.0,
    5,
)

(
    learning_sizes,
    learning_train_scores,
    learning_validation_scores,
) = learning_curve(
    estimator=final_model,
    X=X_train,
    y=y_train,
    train_sizes=training_sizes,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    shuffle=True,
    random_state=RANDOM_STATE,
)

learning_curve_df = pd.DataFrame({
    "training_examples": learning_sizes,
    "training_f1":
        learning_train_scores.mean(axis=1),
    "validation_f1":
        learning_validation_scores.mean(axis=1),
    "validation_f1_std":
        learning_validation_scores.std(axis=1),
})

display(learning_curve_df.round(4))

,training_examples,training_f1,validation_f1,validation_f1_std
0,64,0.9575,0.7864,0.0739
1,128,0.9604,0.8182,0.0297
2,192,0.9584,0.8414,0.0271
3,256,0.9596,0.8358,0.0615
4,320,0.9503,0.8348,0.0492


In [16]:
# plot Learning Curve

learning_figure = px.line(
    learning_curve_df,
    x="training_examples",
    y=[
        "training_f1",
        "validation_f1",
    ],
    markers=True,
    title=f"Learning Curve — {FINAL_MODEL_NAME}",
)

learning_figure.show()

learning_figure.write_html(
    CHART_DIR / "final_model_learning_curve.html"
)

In [17]:
# TEnsorflow Ann Decriptive Comparition

ann_metrics_file = (
    MODEL_DIR / "ann_metrics.json"
)

if ann_metrics_file.exists():
    with open(
        ann_metrics_file,
        "r",
        encoding="utf-8",
    ) as file:
        ann_metrics = json.load(file)

    comparison_df = pd.DataFrame([
        {
            "model": FINAL_MODEL_NAME,
            "macro_f1":
                final_test_metrics["f1_macro"],
            "balanced_accuracy":
                final_test_metrics[
                    "balanced_accuracy"
                ],
            "roc_auc":
                final_test_metrics["roc_auc"],
            "selection_status":
                "Repeated-CV selected",
        },
        {
            "model": "TensorFlow ANN",
            "macro_f1":
                ann_metrics["f1_macro"],
            "balanced_accuracy":
                ann_metrics[
                    "balanced_accuracy"
                ],
            "roc_auc":
                ann_metrics["roc_auc"],
            "selection_status":
                "Descriptive holdout comparison",
        },
    ])

    display(comparison_df.round(4))

,model,macro_f1,balanced_accuracy,roc_auc,selection_status
0,Decision Tree,0.8043,0.8119,0.8556,Repeated-CV selected
1,TensorFlow ANN,0.7993,0.8268,0.9113,Descriptive holdout comparison


In [18]:
# Save Final Predictions

final_prediction_df = X_test.reset_index(
    drop=True
).copy()

final_prediction_df["actual"] = (
    y_test.reset_index(drop=True)
)

final_prediction_df["prediction"] = (
    final_predictions
)

final_prediction_df[
    "probability_class_1"
] = final_probabilities

final_prediction_df.to_csv(
    PREDICTION_DIR / "final_model_predictions.csv",
    index=False,
)

display(final_prediction_df.head(10))

,age,income,region,visits,satisfaction,actual,prediction,probability_class_1
0,44,24497,South,11,4.1,1,1,1.000000
1,33,68313,East,12,2.1,1,0,0.392500
2,58,97385,South,9,3.6,1,1,1.000000
3,65,102494,East,1,1.4,0,0,0.000000
4,33,31307,South,8,1.2,0,0,0.000000
5,54,73768,South,16,1.9,1,1,1.000000
6,52,95009,East,4,1.3,0,0,0.092571
7,51,85960,South,12,1.5,1,1,1.000000
8,38,94401,West,19,4.1,1,1,1.000000
9,63,127974,South,15,3.4,1,1,1.000000


In [19]:
# Save Final model and results

joblib.dump(
    final_model,
    MODEL_DIR / "final_selected_model.joblib",
)

robust_results_df.to_csv(
    MODEL_DIR / "robust_validation_results.csv",
    index=False,
)

statistical_comparison_df.to_csv(
    MODEL_DIR / "model_statistical_comparisons.csv",
    index=False,
)

learning_curve_df.to_csv(
    MODEL_DIR / "final_model_learning_curve.csv",
    index=False,
)

final_selection_summary = {
    "final_model": FINAL_MODEL_NAME,
    "selection_method":
        "Highest lower 95% confidence bound "
        "of repeated-CV macro F1",
    "cross_validation_folds": FOLDS,
    "cross_validation_repeats": REPEATS,
    "total_validation_evaluations":
        FOLDS * REPEATS,
    "holdout_metrics": {
        key: float(value)
        for key, value
        in final_test_metrics.items()
    },
    "bootstrap_f1_interval": {
        "lower": float(bootstrap_lower),
        "upper": float(bootstrap_upper),
    },
    "test_set_note":
        "Previously inspected; metrics are descriptive.",
}

with open(
    MODEL_DIR / "final_selection_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_selection_summary,
        file,
        indent=4,
    )

print("Final model and evaluation saved.")

Final model and evaluation saved.


In [20]:
# Verifird Saved Files

required_files = [
    "final_selected_model.joblib",
    "robust_validation_results.csv",
    "model_statistical_comparisons.csv",
    "final_model_learning_curve.csv",
    "final_selection_summary.json",
]

for filename in required_files:
    path = MODEL_DIR / filename
    print(f"{filename}: {path.exists()}")

prediction_path = (
    PREDICTION_DIR
    / "final_model_predictions.csv"
)

print(
    f"{prediction_path.name}:",
    prediction_path.exists(),
)

final_selected_model.joblib: True
robust_validation_results.csv: True
model_statistical_comparisons.csv: True
final_model_learning_curve.csv: True
final_selection_summary.json: True
final_model_predictions.csv: True


In [21]:
# imports

from sklearn.calibration import (
    CalibratedClassifierCV,
    calibration_curve,
)

from sklearn.metrics import brier_score_loss

In [22]:
# Fit Sigmoid calibrated model

calibrated_final_model = CalibratedClassifierCV(
    estimator=clone(final_model),
    method="sigmoid",
    cv=5,
    n_jobs=-1,
)

calibrated_final_model.fit(
    X_train,
    y_train,
)

calibrated_probabilities = (
    calibrated_final_model.predict_proba(
        X_test
    )[:, 1]
)

calibrated_predictions = (
    calibrated_probabilities >= 0.5
).astype(int)

print("Calibrated model fitted successfully.")

Calibrated model fitted successfully.


In [23]:
# Compare raw and calibrated probabilities

probability_comparison = pd.DataFrame([
    {
        "model": "Raw Decision Tree",
        "accuracy": accuracy_score(
            y_test,
            final_predictions,
        ),
        "balanced_accuracy":
            balanced_accuracy_score(
                y_test,
                final_predictions,
            ),
        "macro_f1": f1_score(
            y_test,
            final_predictions,
            average="macro",
        ),
        "roc_auc": roc_auc_score(
            y_test,
            final_probabilities,
        ),
        "log_loss": log_loss(
            y_test,
            final_probabilities,
        ),
        "brier_score": brier_score_loss(
            y_test,
            final_probabilities,
        ),
        "minimum_probability":
            final_probabilities.min(),
        "maximum_probability":
            final_probabilities.max(),
    },
    {
        "model": "Calibrated Decision Tree",
        "accuracy": accuracy_score(
            y_test,
            calibrated_predictions,
        ),
        "balanced_accuracy":
            balanced_accuracy_score(
                y_test,
                calibrated_predictions,
            ),
        "macro_f1": f1_score(
            y_test,
            calibrated_predictions,
            average="macro",
        ),
        "roc_auc": roc_auc_score(
            y_test,
            calibrated_probabilities,
        ),
        "log_loss": log_loss(
            y_test,
            calibrated_probabilities,
        ),
        "brier_score": brier_score_loss(
            y_test,
            calibrated_probabilities,
        ),
        "minimum_probability":
            calibrated_probabilities.min(),
        "maximum_probability":
            calibrated_probabilities.max(),
    },
])

display(probability_comparison.round(4))

,model,accuracy,balanced_accuracy,macro_f1,roc_auc,log_loss,brier_score,minimum_probability,maximum_probability
0,Raw Decision Tree,0.81,0.8119,0.8043,0.8556,4.1084,0.1613,0.000,1.0000
1,Calibrated Decision Tree,0.86,0.8806,0.8586,0.9151,0.3571,0.1071,0.203,0.8824


In [24]:
# Calibrated Classification Report

print(
    classification_report(
        y_test,
        calibrated_predictions,
        zero_division=0,
    )
)

calibrated_confusion_values = confusion_matrix(
    y_test,
    calibrated_predictions,
)

calibrated_confusion_df = pd.DataFrame(
    calibrated_confusion_values,
    index=[
        f"Actual {label}"
        for label in sorted(y_test.unique())
    ],
    columns=[
        f"Predicted {label}"
        for label in sorted(y_test.unique())
    ],
)

display(calibrated_confusion_df)

              precision    recall  f1-score   support

           0       0.75      0.97      0.84        39
           1       0.98      0.79      0.87        61

    accuracy                           0.86       100
   macro avg       0.86      0.88      0.86       100
weighted avg       0.89      0.86      0.86       100



,Predicted 0,Predicted 1
Actual 0,38,1
Actual 1,13,48


In [25]:
# Calibration Curve

raw_fraction_positive, raw_mean_predicted = (
    calibration_curve(
        y_test,
        final_probabilities,
        n_bins=10,
        strategy="quantile",
    )
)

(
    calibrated_fraction_positive,
    calibrated_mean_predicted,
) = calibration_curve(
    y_test,
    calibrated_probabilities,
    n_bins=10,
    strategy="quantile",
)

raw_calibration_df = pd.DataFrame({
    "predicted_probability":
        raw_mean_predicted,
    "observed_frequency":
        raw_fraction_positive,
    "model": "Raw Decision Tree",
})

calibrated_curve_df = pd.DataFrame({
    "predicted_probability":
        calibrated_mean_predicted,
    "observed_frequency":
        calibrated_fraction_positive,
    "model": "Calibrated Decision Tree",
})

perfect_calibration_df = pd.DataFrame({
    "predicted_probability": [0, 1],
    "observed_frequency": [0, 1],
    "model": "Perfect calibration",
})

calibration_plot_df = pd.concat(
    [
        raw_calibration_df,
        calibrated_curve_df,
        perfect_calibration_df,
    ],
    ignore_index=True,
)

calibration_figure = px.line(
    calibration_plot_df,
    x="predicted_probability",
    y="observed_frequency",
    color="model",
    markers=True,
    title="Probability Calibration Curve",
)

calibration_figure.show()

calibration_figure.write_html(
    CHART_DIR / "probability_calibration.html"
)

In [26]:
# Save Calibrated Predictions

calibrated_prediction_df = (
    X_test.reset_index(drop=True).copy()
)

calibrated_prediction_df["actual"] = (
    y_test.reset_index(drop=True)
)

calibrated_prediction_df["prediction"] = (
    calibrated_predictions
)

calibrated_prediction_df[
    "probability_class_1"
] = calibrated_probabilities

calibrated_prediction_df.to_csv(
    PREDICTION_DIR
    / "calibrated_model_predictions.csv",
    index=False,
)

display(calibrated_prediction_df.head(10))

,age,income,region,visits,satisfaction,actual,prediction,probability_class_1
0,44,24497,South,11,4.1,1,1,0.882422
1,33,68313,East,12,2.1,1,0,0.433327
2,58,97385,South,9,3.6,1,1,0.882422
3,65,102494,East,1,1.4,0,0,0.433219
4,33,31307,South,8,1.2,0,0,0.352190
5,54,73768,South,16,1.9,1,1,0.882422
6,52,95009,East,4,1.3,0,0,0.277685
7,51,85960,South,12,1.5,1,1,0.882422
8,38,94401,West,19,4.1,1,1,0.882422
9,63,127974,South,15,3.4,1,1,0.882422


In [27]:
# Save Calibated Model

CALIBRATED_MODEL_PATH = (
    MODEL_DIR / "final_calibrated_model.joblib"
)

joblib.dump(
    calibrated_final_model,
    CALIBRATED_MODEL_PATH,
)

calibrated_metrics = {
    "base_model": FINAL_MODEL_NAME,
    "calibration_method": "sigmoid",
    "calibration_folds": 5,
    "accuracy": float(
        accuracy_score(
            y_test,
            calibrated_predictions,
        )
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(
            y_test,
            calibrated_predictions,
        )
    ),
    "macro_f1": float(
        f1_score(
            y_test,
            calibrated_predictions,
            average="macro",
        )
    ),
    "roc_auc": float(
        roc_auc_score(
            y_test,
            calibrated_probabilities,
        )
    ),
    "log_loss": float(
        log_loss(
            y_test,
            calibrated_probabilities,
        )
    ),
    "brier_score": float(
        brier_score_loss(
            y_test,
            calibrated_probabilities,
        )
    ),
}

with open(
    MODEL_DIR / "calibrated_model_metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        calibrated_metrics,
        file,
        indent=4,
    )

deployment_summary = {
    "deployment_model":
        "final_calibrated_model.joblib",
    "explanation_model":
        "final_selected_model.joblib",
    "base_algorithm": FINAL_MODEL_NAME,
    "probability_calibration": "sigmoid",
    "decision_threshold": 0.5,
    "note": (
        "Use the calibrated model for predictions. "
        "Use the base Decision Tree for SHAP and "
        "decision-rule explanations."
    ),
}

with open(
    MODEL_DIR / "deployment_model_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        deployment_summary,
        file,
        indent=4,
    )

print("Calibrated model saved:", CALIBRATED_MODEL_PATH)

Calibrated model saved: e:\Practice_PROJECTS\BI_Intelligence\artifacts\models\final_calibrated_model.joblib


In [28]:
# Verify Files

required_calibration_files = [
    MODEL_DIR / "final_calibrated_model.joblib",
    MODEL_DIR / "calibrated_model_metrics.json",
    MODEL_DIR / "deployment_model_summary.json",
    PREDICTION_DIR
    / "calibrated_model_predictions.csv",
]

for path in required_calibration_files:
    print(f"{path.name}: {path.exists()}")

final_calibrated_model.joblib: True
calibrated_model_metrics.json: True
deployment_model_summary.json: True
calibrated_model_predictions.csv: True
